# Actividad 3

Equipo: Panditas Panaderos

## Integrantes
- Emilio Eduardo Garza RIvera - 2976048
- Adrián Alonso Jair Morales Márquez - 3110244
- Miguel Omar Muñoz Solis - 2953688
- Artur Emmanuel Martínez Martínez - 3098530
- Leonardo Yahir Martínez Estrada - 07000080

## Slogan
"A hornear"

In [ ]:
%pip install scikit-learn pandas

# Paso 1: Elección y descripción del dataset

Se seleccionará el dataset de Breast Cancer recuperado del siguiente enlace: *https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html*

* **Nombre:** *Breast Cancer*
* **Fuente:** *scikit-learn developers. (s.f.). load_breast_cancer.*
* **Liga de acceso:** *https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html*

**Contexto**

Para esta actividad se aplicará un modelo de máquina de soporte vectorial y una técnica de agrupamiento sobre un mismo conjunto de datos con el propósito de:
- comparar las preguntas que responde cada enfoque
- evaluar sus resultados
- interpretar los patrones identificados mediante visualizaciones y métricas pertinentes

*Cada registro representa una muestra de tejido mamario obtenida mendiante una aspiración con aguja fina de una masa sospechosa en un paciente.*

*La variable objetivo es la clasificación del tumor. Indica si la masa es benigna o maligna basándose en el análisis de las células.*

*Las variables predictoras contienen características numéricas calculadas a partir de una imagen digitalizada de la muestra de tejido. Incluyen medidas de los núcleos celulares presentes en la imagen, como el radio, la textura, el perímetro, el área, la suavidad, la compacidad, la concavidad, entre otros.*

**Métrica principal de evaluación**

*Se utilizará el **Recall** y el **F1-Score** como métricas principales. El **accuracy** por sí solo puede ser engañoso. En un contexto de diagnóstico médico, un Falso Negativo (un tumor realmente maligno que el modelo clasifica como benigno) es el error más crítico, ya que implicaría que a una paciente con cáncer no se le detecta a tiempo.*

# Paso 2: Preparación del Dataset

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print(f"Dataset Cargado: {X.shape[0]} muestras, {X.shape[1]} características.")

print("\nTipos de datos de cada variable:\n", X.dtypes)

print("\nPresencia de valores faltantes:\n", X.isnull().sum().sum())

print("\nNúmero de registros duplicados:", X.duplicated().sum())

print(f"\nDistribución de clases:\n{y.value_counts(normalize=True) * 100}")

Debido a que no hay datos nulos ni duplicados, procedemos a la partición

In [ ]:
SEED = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Paso 3: Algoritmos de ensamble

In [ ]:
pipeline_base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(random_state=SEED))
])

t0 = time.time()
pipeline_base.fit(X_train, y_train)
t_base = time.time() - t0

y_pred_base = pipeline_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)

print(f"\n[Modelo Base] Características usadas: {X_train.shape[1]} | F1-Score: {f1_base:.4f} | Accuracy: {acc_base:.4f} | Tiempo: {t_base:.4f}s")

# Paso 4: Selección de Caracteristicas

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculamos la matriz de correlación usando solo los datos de entrenamiento
plt.figure(figsize=(12, 10))
matriz_corr = X_train.corr()

# Dibujamos el mapa de calor
sns.heatmap(matriz_corr, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlación de Variables (X_train)')
plt.show()

In [ ]:
# Definimos cuántas variables queremos conservar
K_MEJORES = 15

# 1. Creamos el nuevo Pipeline integrando SelectKBest
pipeline_fs = Pipeline([
    ('scaler', StandardScaler()),
    ('feature_selection', SelectKBest(score_func=f_classif, k=K_MEJORES)),
    ('clf', RandomForestClassifier(random_state=SEED))
])

# 2. Entrenamos el Pipeline
t0 = time.time()
pipeline_fs.fit(X_train, y_train)
t_fs = time.time() - t0

# 3. Predicciones y Evaluación
y_pred_fs = pipeline_fs.predict(X_test)
acc_fs = accuracy_score(y_test, y_pred_fs)
f1_fs = f1_score(y_test, y_pred_fs)

# Mostramos resultados para comparar con el modelo base del Paso 3
print(f"\n[Modelo con Selección] Características usadas: {K_MEJORES} de 30")
print(f"F1-Score: {f1_fs:.4f} | Accuracy: {acc_fs:.4f} | Tiempo: {t_fs:.4f}s")

In [ ]:
# Obtener los nombres de las 15 características seleccionadas (Parte extra)
variables_elegidas = pipeline_fs.named_steps['feature_selection'].get_feature_names_out(X.columns)
print("Variables seleccionadas:", variables_elegidas)

# Paso 5: Análisis de Interpretabilidad de los Datos

- Para reducir los datos usamos SelectKBest con el criterio estadístico f_classif (ANOVA). Esto significa que el algoritmo evaluó cada una de las variables originales y les puso una 'calificación' basada en qué tan buenas son para distinguir entre un tumor maligno y uno benigno.
    
    Decidimos ser estrictos y conservar solo las 15 características con la puntuación más alta. El efecto que logramos con esto fue limpiar la información repetida; le quitamos ruido al modelo para que sea más rápido y eficiente, y comprobamos que no perdimos nada de precisión en las predicciones.

- Como utilizamos un método de selección directa, las variables no se alteraron. Las 15 características "sobrevivientes" que el algoritmo consideró más importantes para predecir el cáncer son:
    
    1. mean radius, 2. mean perimeter, 3. mean area, 4. mean compactness, 5. mean concavity, 6. mean concave points, 7. radius error, 8. perimeter error, 9. area error, 10. worst radius, 11. worst perimeter, 12. worst area, 13. worst compactness, 14. worst concavity, 15. worst concave points.

- Para este proyecto decidimos no utilizar PCA. En un contexto médico como el diagnóstico de cáncer, es de vital importancia que los doctores puedan interpretar los datos (por ejemplo, saber que el 'radio promedio' del tumor es alarmante).

    Si hubiéramos aplicado PCA, las variables se habrían comprimido en componentes matemáticos abstractos (perdiendo sus nombres y significado original), volviendo al modelo una 'caja negra' imposible de interpretar para un médico. Por eso SelectKBest fue la opción ideal.

# Paso 6: 

| Configuración | Número de características | Métrica principal (F1-Score) | Métrica complementaria (Accuracy) | Tiempo de entrenamiento |
|---|---|---|---|---|
| Modelo base | 30 | 0.9655 | 0.9561 | 0.0765s |
| Modelo con selección o reducción | 15 | 0.9655 | 0.9561 | 0.0645s |

Al comparar ambas configuraciones, el F1-Score y el Accuracy se mantienen exactamente iguales (0.9655 y 0.9561) a pesar de reducir a la mitad el número de características, de 30 a 15. Esto indica que las 15 variables descartadas no aportaban información relevante para distinguir entre tumores malignos y benignos. Además, el tiempo de entrenamiento disminuyó de 0.0765s a 0.0645s, aproximadamente un 16% mas rápido. Por lo tanto, la selección de características logró simplificar el modelo sin sin sacrificar desempeño, resultando en un modelo más eficiente y más fácil de interpretar, sin ninguna pérdida importante de información

# Paso 7:

In [ ]:
# Búsqueda de Hiperparámetros

# 1. Definimos el espacio de búsqueda
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5, 10]
}

# 2. Configuramos la validación cruzada estratificada
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# 3. Usamos el mismo pipeline con selección de características
grid_search = GridSearchCV(
    estimator=pipeline_fs,
    param_grid=param_grid,
    scoring='f1',
    cv=cv_strategy,
    n_jobs=-1
)

# 4. Entrenamos solo con datos de entrenamiento
t0 = time.time()
grid_search.fit(X_train, y_train)
t_grid = time.time() - t0

print(f"Mejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor F1-Score (validación cruzada): {grid_search.best_score_:.4f}")
print(f"Tiempo de búsqueda: {t_grid:.2f}s")

# 5. Evaluamos el mejor modelo en el conjunto de prueba
best_model = grid_search.best_estimator_
y_pred_grid = best_model.predict(X_test)
f1_grid = f1_score(y_test, y_pred_grid)
acc_grid = accuracy_score(y_test, y_pred_grid)

print(f"\n[Modelo Optimizado] F1-Score: {f1_grid:.4f} | Accuracy: {acc_grid:.4f}")

| Hiperparámetro | Valores explorados | Mejor valor |
|---|---|---|
| n_estimators | 100, 200, 300 | 200 |
| max_depth | None, 5, 10 | None |
| min_samples_split | 2, 5, 10 | 2 |

La busqueda con GridSearchCV exploró combinaciones de n_estimators, max_depth y min_samples_split, usando validación cruzada estratificada de 5 particiones sobre el conjunto de entrenamiento. El mejor resultado en validación cruzada fue un F1-Score de 0.9582, obteniendo con max_depth=None,min_samples_split=2 y n_estimators=200. Al evaluar este modelo optimizado sobre el conjunto de prueba, se obtuvo un F1-Score de 0.9655 y un Accuracy de 0.9561, vlaroes idénticos a los del modelo del paso 4 sin optimización de hiperparámetros. Esto indica que la configuración por defecto del Random Forest ya se encontraba cerca del óptimo para este problema, y que el dataset de cáncer de mama es relativamente sencillo de separar con las varibales seleccionadas, dejando poco margen de mejora mediante el ajuste de hiperparámetros. La diferencia entre el F1-Score de validación cruzada (0.9582) y el de prueba (0.9655) es esperada, ya que el primero promedia el desempeño sobre varias particiones distintas, mientras que el segundo corresponde a una sola partición fija de prueba